Notebook 03 - Feature Engineering
NLP Assignment - Fake News Detection
Person 3: N.A.Matheesha Desaman (CIT-24-01-0435)

# Feature Engineering — BERT Embeddings
**Why BERT?** Unlike TF-IDF or Word2Vec, BERT understands context.
The word "bank" means different things in "river bank" vs "bank account".
BERT captures this — making it ideal for detecting fake news where
misleading context is the key signal.

In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertModel
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported!")
print("PyTorch version:", torch.__version__)

✅ Libraries imported!
PyTorch version: 2.12.1+cpu


In [2]:
df = pd.read_csv('../data/welFake_cleaned.csv')
print("Shape:", df.shape)
df.head()

Shape: (72095, 2)


,final_text,label
0,law enforcement high alert following threat co...,1
1,post vote hillary already,1
2,unbelievable obamas attorney general say charl...,1
3,bobby jindal raised hindu us story christian c...,0
4,satan russia unvelis image terrifying new supe...,1


In [3]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print("✅ BERT tokenizer loaded!")
print("Vocabulary size:", tokenizer.vocab_size)

✅ BERT tokenizer loaded!
Vocabulary size: 30522


In [4]:
sample = df['final_text'][0][:200]
print("Original text:", sample)

tokens = tokenizer(sample, padding=True, truncation=True, max_length=128, return_tensors='pt')
print("\nToken IDs shape:", tokens['input_ids'].shape)
print("Sample token IDs:", tokens['input_ids'][0][:15])

Original text: law enforcement high alert following threat cop white blacklivesmatter fyf terrorist video comment expected barack obama member fyf fukyoflag blacklivesmatter movement called lynching hanging white pe

Token IDs shape: torch.Size([1, 44])
Sample token IDs: tensor([  101,  2375,  7285,  2152,  9499,  2206,  5081,  8872,  2317,  2304,
         3669,  6961, 18900,  3334,  1042])


## BERT Embedding Strategy
Because BERT is computationally expensive, we use two approaches:
1. **For ML model (Naive Bayes):** We extract BERT embeddings for a sample 
   of the data and use them as features.
2. **For DL model (BERT fine-tuning):** We fine-tune BERT directly 
   on the classification task in the model notebook.

This is the standard real-world approach for BERT-based NLP systems.

In [6]:
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()

def get_bert_embedding(text):
    inputs = tokenizer(text, padding=True, truncation=True, 
                       max_length=128, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)
    # Use [CLS] token embedding as sentence representation
    return outputs.last_hidden_state[:, 0, :].squeeze().numpy()

# Use a sample of 2000 records (full 72k would take hours on CPU)
sample_df = df.sample(n=2000, random_state=42).reset_index(drop=True)

print("Extracting BERT embeddings for 2000 samples...")
print("This will take several minutes. Please wait!")

embeddings = []
for i, text in enumerate(sample_df['final_text']):
    emb = get_bert_embedding(str(text))
    embeddings.append(emb)
    if (i+1) % 200 == 0:
        print(f"  Processed {i+1}/2000...")

X_bert = np.array(embeddings)
y_bert = sample_df['label'].values

print(f"\n✅ Done! Embeddings shape: {X_bert.shape}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting BERT embeddings for 2000 samples...
This will take several minutes. Please wait!
  Processed 200/2000...
  Processed 400/2000...
  Processed 600/2000...
  Processed 800/2000...
  Processed 1000/2000...
  Processed 1200/2000...
  Processed 1400/2000...
  Processed 1600/2000...
  Processed 1800/2000...
  Processed 2000/2000...

✅ Done! Embeddings shape: (2000, 768)


In [7]:
np.save('../data/bert_embeddings_X.npy', X_bert)
np.save('../data/bert_labels_y.npy', y_bert)

print("✅ BERT embeddings saved!")
print("Shape:", X_bert.shape)

✅ BERT embeddings saved!
Shape: (2000, 768)
